<a href="https://colab.research.google.com/github/AlexisCuevasUriostique/AufhebenAdapter/blob/main/InvariantDifferential.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import torch
from google.colab import drive

if not os.path.exists('/content/drive'):
    print("Mounting Google Drive...")
    drive.mount('/content/drive')
else:
    print("Google Drive already mounted.")

device = "cuda" if torch.cuda.is_available() else "cpu"

# Re-defined with clean strings to avoid mobile keyboard artifacts
BASE_CHECKPOINT = "/content/drive/MyDrive/Aufheben/checkpoints/hegelian_step_300_continued.pt"
ADAPTER_PATH = "/content/drive/MyDrive/Aufheben/adapters/hegelian_flood_adapter.pth"

# Verification check
print(f"Checking for Base Checkpoint: {'Found' if os.path.exists(BASE_CHECKPOINT) else 'NOT FOUND'}")
print(f"Checking for Adapter: {'Found' if os.path.exists(ADAPTER_PATH) else 'NOT FOUND'}")

Google Drive already mounted.
Checking for Base Checkpoint: Found
Checking for Adapter: Found


In [ ]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig
from datasets import load_dataset
import json
import datetime
import os
from tqdm import tqdm

# --- 1. Hardware Precision Override ---
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

class HybridInterpretabilityLens(nn.Module):
    def __init__(self, hidden_size=2560):
        super().__init__()
        self.thesis_adapter = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.GELU(),
            nn.Linear(hidden_size // 2, hidden_size),
            nn.Dropout(0.1)
        )
        self.antithesis_scorer = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 4),
            nn.GELU(),
            nn.Linear(hidden_size // 4, 1),
            nn.Sigmoid()
        )
        self.anstoss_scorer = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 4),
            nn.GELU(),
            nn.Linear(hidden_size // 4, 1),
            nn.Sigmoid()
        )
        self.gate = nn.Linear(hidden_size * 2, hidden_size)

    def forward(self, x):
        x = x.to(self.thesis_adapter[0].weight.dtype)
        p_ctx = self.thesis_adapter(x)
        contradiction = self.antithesis_scorer(x)
        impossibility = self.anstoss_scorer(x)
        friction_scalar = (contradiction + impossibility) / 2.0
        n_ctx = friction_scalar * x
        gate_in = torch.cat([p_ctx, n_ctx], dim=-1)
        g = torch.sigmoid(self.gate(gate_in))
        logic_vector = p_ctx * g - (n_ctx * (1 - g))
        return {"friction_scalar": friction_scalar, "negative_context": n_ctx, "logic_vector": logic_vector}

# --- 4. Model & Lens Setup ---
tokenizer = AutoTokenizer.from_pretrained("microsoft/phi-2")
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token

# Fix: Explicitly set pad_token_id in the config before loading the model
config = AutoConfig.from_pretrained("microsoft/phi-2")
config.pad_token_id = tokenizer.pad_token_id

model = AutoModelForCausalLM.from_pretrained("microsoft/phi-2", config=config, torch_dtype=torch.float16).to(device)
print("Loading base Phi-2 weights...")
checkpoint = torch.load(BASE_CHECKPOINT, map_location='cpu')
state_dict = checkpoint['model_state_dict'] if 'model_state_dict' in checkpoint else checkpoint
new_state_dict = {k.replace("base.model.", "model.").replace("base.", ""): v for k, v in state_dict.items()}
model.load_state_dict(new_state_dict, strict=False)

lens = HybridInterpretabilityLens(hidden_size=2560).to(device)
lens_sd = lens.state_dict()
s300_weights = torch.load(BASE_CHECKPOINT, map_location=device)
s300_sd = s300_weights['model_state_dict'] if 'model_state_dict' in s300_weights else s300_weights
for k in s300_sd:
    if k.startswith(("thesis_adapter.", "antithesis_scorer.", "anstoss_scorer.")) and k in lens_sd: lens_sd[k] = s300_sd[k]

flood_weights = torch.load(ADAPTER_PATH, map_location=device)
for k in flood_weights:
    if "gate" in k and k in lens_sd: lens_sd[k] = flood_weights[k]

lens.load_state_dict(lens_sd)
lens.float().eval()
model.eval()

# --- 5. Diagnostic Run (Local JSON Override) ---
THRESHOLD = 4.0

# Define your input and output paths directly on the mounted Drive
INPUT_FILE = "/content/drive/MyDrive/truthfulqa_hallucination_poc (2).json"
OUTPUT_FILE = "/content/drive/MyDrive/point_hallucination_poc.json"

print(f"Bypassing HF Hub. Loading prompts directly from local JSON: {INPUT_FILE}")

# Parse the JSON directly
with open(INPUT_FILE, "r") as f:
    local_tqa_data = json.load(f)

results = []

print("Running structural diagnostic on local TruthfulQA prompts...")
# Iterate over the parsed JSON array
for item in tqdm(local_tqa_data):
    prompt = item["prompt"]
    idx_str = item["id"] # Preserve the original TQA-X identifier

    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model(inputs.input_ids, output_hidden_states=True)
        base_hidden = outputs.hidden_states[-1]

        # Lens handles f32 internally (TF32 disabled)
        lens_outputs = lens(base_hidden.float())
        n_ctx = lens_outputs["negative_context"]
        logic_vector = lens_outputs["logic_vector"]
        friction_scalar = lens_outputs["friction_scalar"]

        # Measure the structural tension of the logic vector directly
        token_norms = torch.norm(logic_vector, p=2, dim=-1).squeeze(0)

        anomalies = []
        for i, norm in enumerate(token_norms):
            norm_val = norm.item()
            if norm_val > THRESHOLD:
                token_id = inputs.input_ids[0, i].item()
                token_str = tokenizer.decode([token_id])

                # The friction scalar replaces the attention matrix row
                scalar_val = friction_scalar[0, i, 0].item()

                # Projection back to vocab using f32 for the logit lens slice
                peak_vec = n_ctx[0, i, :]
                logits = torch.matmul(model.lm_head.weight.float(), peak_vec)
                top_ids = torch.topk(logits, 5).indices
                decoded_concepts = [tokenizer.decode([t_id]) for t_id in top_ids]

                anomalies.append({
                    "token_index": i,
                    "token_str": token_str,
                    "activation_score": norm_val,
                    "friction_scalar": scalar_val,
                    "logit_lens_output": decoded_concepts
                })

        results.append({
            "id": idx_str,
            "prompt": prompt,
            "peak_norm": float(torch.max(token_norms).item()),
            "anomalies": anomalies
        })

# --- Save to JSON ---
with open(OUTPUT_FILE, "w") as f:
    json.dump(results, f, indent=4)

print(f"\nExecution complete. Hybrid structural anomaly data saved to {OUTPUT_FILE}")

config.json:   0%|          | 0.00/735 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.34k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/1.08k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Loading base Phi-2 weights...
Bypassing HF Hub. Loading prompts directly from local JSON: /content/drive/MyDrive/truthfulqa_hallucination_poc (2).json
Running structural diagnostic on local TruthfulQA prompts...


100%|██████████| 817/817 [01:33<00:00,  8.71it/s]



Execution complete. Hybrid structural anomaly data saved to /content/drive/MyDrive/hybrid_hallucination_poc.json
